# Creating Cohorts of Songs (Rolling Stones Spotify Dataset)
This notebook guides you through performing exploratory data analysis (EDA) and cluster analysis to group the Rolling Stones' songs into cohorts based on their musical characteristics (audio features).

### Project Objectives:
1. **Data Inspection & Cleaning:** Identify and handle missing values, duplicates, and format audio features.
2. **Exploratory Data Analysis (EDA):** Recommending albums based on popular song count, examining feature correlations, and analyzing temporal trends.
3. **Dimensionality Reduction:** Apply Principal Component Analysis (PCA) to simplify high-dimensional features and visualize the songs in 2D.
4. **Cluster Analysis (Creating Cohorts):** Use WCSS (Elbow) and Silhouette Scores to determine the optimal cluster count, fit a K-Means model, and profile each cohort musically.


In [ ]:
# Detect environment and setup file imports for Google Colab compatibility
import os
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Google Colab environment.')
    print("Please upload the dataset file 'rolling_stones_spotify.csv' when prompted below...")
    from google.colab import files
    uploaded = files.upload()
else:
    print('Running in local Python environment. Path defaults to raw data folder.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set seaborn theme and plot sizes
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Path resilience: checks local directory path first, then Colab upload folder
try:
    df = pd.read_csv('1736848608_rolling_stones_spotify/rolling_stones_spotify.csv', index_col=0)
except FileNotFoundError:
    df = pd.read_csv('rolling_stones_spotify.csv', index_col=0)

print(f'Dataset loaded successfully. Shape (rows , column): {df.shape}')
df

## Step 1: Initial Data Inspection & Data Cleaning

### Why do we perform this step?
* **Data Cleaning** is vital to ensure our model isn't training on duplicate entries or getting corrupted by missing data.
* **Duplicate Checks:** If the same track is included multiple times (such as live re-issues or deluxe compilation track listings), it can bias the cluster sizes.
* **Missing Values:** Machine learning algorithms in scikit-learn will fail to fit if there are missing values (`NaN`).


In [ ]:
# Inspect general information
print('=== Metadata Info ===')
df.info()

print('\n=== Missing Values Check ===')
print(df.isna().sum())

print('\n=== Unique IDs and Duplicates Check ===')
duplicate_ids = df.duplicated(subset=['id']).sum()
print(f'Number of duplicate tracks by Unique Spotify ID: {duplicate_ids}')

# Print descriptive statistics of columns
print('\n=== Summary Statistics (Raw Data) ===')
df.describe()


### Step 1 Observations:
* **Shape:** The dataset contains $1610$ rows (songs) and $17$ columns.
* **Missing Values:** There are no null or missing values across all columns. The dataset is fully complete.
* **Duplicates:** There are $0$ duplicate tracks based on the unique Spotify `id` column. No rows need to be deleted, preserving the integrity of all records.


## Step 2: Exploratory Data Analysis & Feature Engineering

### 2.1 Recommending Albums based on Popular Songs

To find the best albums to recommend to a general audience:
1. We calculate the $75$th percentile of popularity scores across all tracks.
2. We filter the dataset to keep only tracks with a popularity score at or above this threshold.
3. We count the number of popular tracks in each album and recommend the top two albums.


In [ ]:
# Identify the 75th percentile of popularity
popularity_75th = df['popularity'].quantile(0.75)
print(f'75th Percentile Popularity Threshold: {popularity_75th}')

# Filter popular songs and group by album
popular_tracks = df[df['popularity'] >= popularity_75th]
album_popular_counts = popular_tracks.groupby('album')['name'].count().sort_values(ascending=False)

# Plot the top albums
sns.barplot(x=album_popular_counts.head(10).values, y=album_popular_counts.head(10).index, palette='viridis')
plt.title(f'Top 10 Albums by Count of Popular Songs (Popularity >= {popularity_75th})')
plt.xlabel('Count of Popular Tracks')
plt.ylabel('Album Name')
plt.tight_layout()
plt.show()

print('\n=== Top 3 Albums by Popular Songs Count ===')
print(album_popular_counts.head(3))


### Step 2.1 Observations & Recommendations:
* **Popularity Threshold:** The $75$th percentile threshold is **27.0** (out of 100). Songs with scores $\ge 27$ are considered 'popular' relative to the rest of this Rolling Stones catalog.
* **Top Albums:** The top albums with the highest number of popular songs are **Honk (Deluxe)**, **Exile On Main Street (2010 Re-Mastered)**, and **Exile On Main Street (Deluxe Version)** (each contains $18$ popular tracks).
* **Recommendation:** We recommend recommending **Honk (Deluxe)** and **Exile On Main Street (2010 Re-Mastered)** to anyone looking for the most popular songs from the band.


### 2.2 Correlation Analysis: Popularity vs. Audio Features

To examine what song features correlate with popularity, we will compute Pearson correlation coefficients ($r$) between popularity and other numeric features, then plot them using a heatmap.


In [ ]:
# Compute correlation matrix for numeric columns
numeric_cols = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence', 'popularity', 'duration_ms']
corr_matrix = df[numeric_cols].corr()

# Visualize using Seaborn heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5)
plt.title('Correlation Matrix of Song Popularity and Audio Features')
plt.tight_layout()
plt.show()

print('\nCorrelation of Features with Popularity (Sorted):')
print(corr_matrix['popularity'].sort_values(ascending=False))


### Step 2.2 Observations:
* **Positive Correlations:** `loudness` ($0.156$) and `danceability` ($0.141$) show the strongest positive correlation with popularity. Studio tracks that are louder, more upbeat, and easier to dance to tend to be more popular.
* **Negative Correlations:** `liveness` ($-0.206$) and `speechiness` ($-0.137$) show the strongest negative correlation with popularity. This suggests live concert tracks or talkative/spoken tracks are less popular with general Spotify listeners.


### 2.3 Temporal Trend Analysis: Evolution of Songs Over Time

We extract the release year from the release dates and see how song features and average popularity have changed across the years.


In [ ]:
# Convert release_date to datetime and extract year
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Group by release year and compute mean metrics
yearly_trends = df.groupby('release_year')[['popularity', 'energy', 'acousticness', 'loudness']].mean().reset_index()

# Plot trend evolution
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Release Year')
ax1.set_ylabel('Average Popularity', color='tab:blue')
sns.lineplot(data=yearly_trends, x='release_year', y='popularity', ax=ax1, color='tab:blue', label='Popularity', marker='o')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Average Feature Metric', color='tab:red')
sns.lineplot(data=yearly_trends, x='release_year', y='energy', ax=ax2, color='tab:red', label='Energy', marker='s')
sns.lineplot(data=yearly_trends, x='release_year', y='acousticness', ax=ax2, color='tab:orange', label='Acousticness', marker='^')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Evolution of Song Popularity, Energy, and Acousticness Over Time (1964 - 2022)')
fig.tight_layout()
plt.show()


### Step 2.3 Observations:
* **Decade Trends:** The band's early 1960s albums had higher acousticness, which dropped off significantly as the band transitioned into heavy blues-rock.
* **Energy and Loudness:** Energy stays consistently high, representing their signature hard-rock sound, while popularity shows massive peaks around major reissue/remaster compilation drops (e.g. 2010 and 2020-2022).


## Step 3: Dimensionality Reduction using PCA

### Why is Dimensionality Reduction significant here?
* **Curse of Dimensionality:** Music contains many features. As the number of dimensions increases, distance metrics begin to converge, making clustering less effective.
* **Feature Standardizing:** We must scale the features using `StandardScaler` to bring them onto the same scale (mean=0, variance=1) before running PCA or K-Means.
* **Variance Preservation:** PCA project the data onto Principal Components, preserving maximum variance in fewer dimensions, allowing us to visualize clusters in 2D.


In [ ]:
# Features to scale and cluster
audio_features = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']
X = df[audio_features]

# Apply standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run PCA
pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Plot cumulative explained variance ratio
cum_variance = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cum_variance)+1), cum_variance, marker='o', linestyle='--', color='purple')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.axhline(y=0.80, color='r', linestyle=':', label='80% Variance Threshold')
plt.legend()
plt.show()

print('Explained variance ratio per principal component:')
for idx, ev in enumerate(pca.explained_variance_ratio_):
    print(f'PC{idx+1}: {ev:.4f} (Cumulative: {cum_variance[idx]:.4f})')


### Step 3 Observations:
* **Variance Explanation:** The first Principal Component (PC1) explains $32.45\%$ of the variance, and the second component (PC2) explains $17.94\%$. Together, they represent $50.38\%$ of the total variance.
* **Dimensionality Reduction Choice:** By selecting the first five components, we explain $81.38\%$ of the variance, which simplifies the dataset while keeping the majority of the info. For visualization, we will project the songs using the first two components (PC1 vs PC2).


In [ ]:
# Plot 2D scatter of the principal components
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, color='teal', edgecolors='w')
plt.title('2D Projection of Rolling Stones Songs using PCA')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.show()


## Step 4: Cluster Analysis

### 4.1 Finding the Optimal Number of Clusters (K)

We use two main metrics to determine the right number of cohorts:
1. **Elbow Method (WCSS):** Measures how tight the clusters are. We look for an 'elbow' where the drop in WCSS slows down.
2. **Silhouette Analysis:** Measures how distinct the clusters are. A higher silhouette coefficient represents better-separated clusters.


In [ ]:
wcss = []
sil_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Plot results side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(k_range, wcss, marker='o', color='tab:blue')
ax1.set_title('Elbow Method (WCSS)')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Within-Cluster Sum of Squares (Inertia)')

ax2.plot(k_range, sil_scores, marker='s', color='tab:orange')
ax2.set_title('Silhouette Score vs. K')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Average Silhouette Coefficient')

plt.tight_layout()
plt.show()


### Step 4.1 Observations:
* **Elbow Curve:** The WCSS drops smoothly, but the rate of decrease slows down around $K=3$ and $K=4$.
* **Silhouette Score:** The silhouette score reaches its peak at $K=2$ ($0.2149$) and remains strong at $K=3$ ($0.1896$).
* **Selection:** We choose **$K=3$** clusters because it balances strong statistical clustering separation with practical, distinct song grouping profiles.


### 4.2 Training K-Means and Visualizing the Cohorts in PCA space

We train the final K-Means model with $K=3$ and project the cluster assignments onto the 2D PCA representation.


In [ ]:
# Run K-Means with K=3
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans_model.fit_predict(X_scaled)

# Project cluster labels onto PC1 and PC2
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'], cmap='viridis', alpha=0.6, edgecolors='w', s=50)
plt.legend(*scatter.legend_elements(), title='Cohorts')
plt.title('Song Cohorts Visualized in 2D PCA Space (K=3)')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.show()


## Step 5: Cohort Profiling & Music Explanations

To understand the musical characteristics of each cluster, we compute the average feature values for each cohort.


In [ ]:
# Calculate cluster means
cohort_profiles = df.groupby('cluster')[audio_features].mean()
print('=== Cohort Feature Averages ===')
print(cohort_profiles.round(3).to_string())

# Visual heatmap comparison of cohorts
sns.heatmap(cohort_profiles, annot=True, cmap='YlGnBu', fmt='.3f', linewidths=0.5)
plt.title('Cohort Musical Profiles (Feature Means)')
plt.ylabel('Cohort (Cluster ID)')
plt.show()

print('\nNumber of Songs per Cohort:')
print(df['cluster'].value_counts())


### Cohort Descriptions & Profile Definitions:

Based on the average feature values, we define and name our three cohorts:

1. **Cohort 0: High-Energy Live Tracks & Concert Recordings (596 songs)**
   * *Sonic Profile:* Very high `liveness` ($0.821$) and `energy` ($0.924$), high `loudness` ($-5.38$ dB), and fast `tempo` ($137.9$ BPM).
   * *Musical Description:* These are live concert versions, audience-filled recordings, and high-energy bootleg tracks capturing the band's stage performance.
   * *Recommendation use case:* Recommend to fans who prefer live stadium atmospheres and energetic concert recordings.

2. **Cohort 1: Upbeat & Danceable Studio Rock Hits (584 songs)**
   * *Sonic Profile:* High `danceability` ($0.564$) and `energy` ($0.821$), exceptionally high `valence` ($0.789$ - happy/cheerful mood), and low `acousticness` ($0.186$).
   * *Musical Description:* These are upbeat, groovy studio rock anthems that convey positivity, happiness, and high rhythmic drive.
   * *Recommendation use case:* Recommend to users looking for cheerful, happy rock music, party playlists, or road trip soundtracks.

3. **Cohort 2: Acoustic, Melodic & Slow Tempo Ballads (430 songs)**
   * *Sonic Profile:* High `acousticness` ($0.430$), moderate `energy` ($0.571$), lower `loudness` ($-9.72$ dB), slower `tempo` ($115.1$ BPM), and low `instrumentalness` ($0.093$).
   * *Musical Description:* These are softer, acoustic-led songs, melodic tracks, and slow tempo ballads with vocal-focused content.
   * *Recommendation use case:* Recommend to listeners seeking a relaxed, chill vibe, acoustic sessions, or slow-listening rock ballads.
